<div style="text-align:left;">
    <span style="
        display:inline-block;
        background-color:#4169E1;
        color:white;
        padding:10px 20px;
        border-radius:8px;
        font-size:45px;
        font-weight:bold;
    ">
        TabICLv2 for EHR Mortality Risk Prediction
    </span>
</div>

**Author:** Sarra Chouk 

**Student ID:** 60300372

**Project:** EHR Mortality Risk Prediction  

**Dataset:** MIMIC-IV

**Date:** April 4, 2026  

---

### **Objective**
To evaluate TabICLv2 on the encounter-level mortality prediction task and compare its performance against previously tested baseline machine learning models.

### **Model Setup and Library Imports**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import gc
import os
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", str((Path.cwd() / "../artifacts/numba_cache").resolve()))

import psutil
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

from tabicl import TabICLClassifier

### **Load Imputed Splits**

In [12]:
IMPUTED_BASE_PATH = "../data/processed/splits_imputed"

X_train = pd.read_csv(f"{IMPUTED_BASE_PATH}/X_train.csv")
y_train = pd.read_csv(f"{IMPUTED_BASE_PATH}/y_train.csv").squeeze("columns")

X_test = pd.read_csv(f"{IMPUTED_BASE_PATH}/X_test.csv")
y_test = pd.read_csv(f"{IMPUTED_BASE_PATH}/y_test.csv").squeeze("columns")

X_deployment = pd.read_csv(f"{IMPUTED_BASE_PATH}/X_deployment.csv")
y_deployment = pd.read_csv(f"{IMPUTED_BASE_PATH}/y_deployment.csv").squeeze("columns")

print("Shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("X_deployment:", X_deployment.shape)
print("y_deployment:", y_deployment.shape)

Shapes:
X_train: (380461, 49)
y_train: (380461,)
X_test: (83356, 49)
y_test: (83356,)
X_deployment: (82211, 49)
y_deployment: (82211,)


### **Data validation and Sanity Checks**

In [13]:
assert len(X_train) == len(y_train), "Train X/y row count mismatch."
assert len(X_test) == len(y_test), "Test X/y row count mismatch."
assert len(X_deployment) == len(y_deployment), "Deployment X/y row count mismatch."

if X_train.isna().sum().sum() > 0:
    raise ValueError("NaNs found in X_train")
if X_test.isna().sum().sum() > 0:
    raise ValueError("NaNs found in X_test")
if X_deployment.isna().sum().sum() > 0:
    raise ValueError("NaNs found in X_deployment")

print("Target distribution:")
print("\nTrain:")
print(y_train.value_counts(normalize=True).rename("proportion"))

print("\nTest:")
print(y_test.value_counts(normalize=True).rename("proportion"))

print("\nDeployment:")
print(y_deployment.value_counts(normalize=True).rename("proportion"))

print("\nAll checks passed.")

Target distribution:

Train:
target
0   0.978289
1   0.021711
Name: proportion, dtype: float64

Test:
target
0   0.978766
1   0.021234
Name: proportion, dtype: float64

Deployment:
target
0   0.978458
1   0.021542
Name: proportion, dtype: float64

All checks passed.


### **Evaluation Helper**

In [ ]:
IMPUTED_BASE_PATH = "../data/processed/splits_imputed"
TABICL_CHUNK_SIZE = 512
TABICL_MAX_TRAIN_ROWS = 50000
TABICL_RANDOM_STATE = 42
TABICL_OFFLOAD_DIR = str((Path.cwd() / "../artifacts/tabicl_offload").resolve())
os.makedirs(TABICL_OFFLOAD_DIR, exist_ok=True)

def print_ram_usage(note=""):
    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / (1024 ** 3)
    print(f"{note} RAM used: {ram_gb:.2f} GB")

def print_df_memory(df, name):
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"{name} memory usage: {mem_mb:.2f} MB")

def make_tabicl_training_context(X, y, max_rows=TABICL_MAX_TRAIN_ROWS, random_state=TABICL_RANDOM_STATE):
    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    if max_rows is None or len(X) <= max_rows:
        print(f"TabICL training context uses all {len(X):,} rows.")
        return X.copy(), y.astype(np.int8, copy=False).copy()

    _, X_ctx, _, y_ctx = train_test_split(
        X,
        y,
        test_size=max_rows,
        stratify=y,
        random_state=random_state,
    )

    X_ctx = X_ctx.sort_index().reset_index(drop=True).copy()
    y_ctx = y_ctx.sort_index().reset_index(drop=True).astype(np.int8, copy=False).copy()

    print(
        f"TabICL training context capped at {len(X_ctx):,}/{len(X):,} rows "
        f"(stratified sample, random_state={random_state})."
    )
    print(f"Positive-class rate full={y.mean():.4%}, context={y_ctx.mean():.4%}")

    return X_ctx, y_ctx

def report_cache_sizes(model):
    cache_dict = getattr(model, "model_kv_cache_", None)
    if not cache_dict:
        print("No TabICL cache was built.")
        return

    total_mb = 0
    for name, cache in cache_dict.items():
        cache_mb = cache.cache_size_mb()
        total_mb += cache_mb
        print(f"Cache[{name}] ({cache.cache_type}): {cache_mb:,} MB")

    print(f"Total cache size: {total_mb:,} MB")

def predict_positive_proba_in_chunks(model, X, chunk_size=TABICL_CHUNK_SIZE, positive_class=1):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be a positive integer.")
    if positive_class not in model.classes_:
        raise ValueError(f"Class {positive_class!r} is not present in model.classes_.")

    class_index = int(np.where(model.classes_ == positive_class)[0][0])
    total_rows = len(X)
    scores = []

    for start in range(0, total_rows, chunk_size):
        stop = min(start + chunk_size, total_rows)
        X_chunk = X.iloc[start:stop] if hasattr(X, "iloc") else X[start:stop]
        chunk_scores = model.predict_proba(X_chunk)[:, class_index].astype(np.float32, copy=False)
        scores.append(chunk_scores)
        print(f"Scored rows {stop:,}/{total_rows:,}")
        print_ram_usage(f"After chunk ending at row {stop:,}")
        gc.collect()

    return np.concatenate(scores, axis=0)

def compute_metrics(model_name, y_true, y_pred, y_score):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "predicted_positives": int((y_pred == 1).sum()),
    }

In [15]:
X_train = pd.read_csv(f"{IMPUTED_BASE_PATH}/X_train.csv").astype(np.float32)
y_train = pd.read_csv(f"{IMPUTED_BASE_PATH}/y_train.csv").squeeze("columns").astype(np.int8)

X_test = pd.read_csv(f"{IMPUTED_BASE_PATH}/X_test.csv").astype(np.float32)
y_test = pd.read_csv(f"{IMPUTED_BASE_PATH}/y_test.csv").squeeze("columns").astype(np.int8)

print("Shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

assert len(X_train) == len(y_train), "Train X/y row count mismatch."
assert len(X_test) == len(y_test), "Test X/y row count mismatch."

if X_train.isna().sum().sum() > 0:
    raise ValueError("NaNs found in X_train")
if X_test.isna().sum().sum() > 0:
    raise ValueError("NaNs found in X_test")

print("\nTarget distribution:")
print("\nTrain:")
print(y_train.value_counts(normalize=True).rename("proportion"))

print("\nTest:")
print(y_test.value_counts(normalize=True).rename("proportion"))

print("\nAll checks passed.")
print_ram_usage("After loading train/test")
print_df_memory(X_train, "X_train")
print_df_memory(X_test, "X_test")
print(f"y_train memory usage: {y_train.memory_usage(deep=True) / (1024 ** 2):.2f} MB")
print(f"y_test memory usage: {y_test.memory_usage(deep=True) / (1024 ** 2):.2f} MB")

X_train_tabicl, y_train_tabicl = make_tabicl_training_context(X_train, y_train)
print("TabICL training context shape:", X_train_tabicl.shape)
print_df_memory(X_train_tabicl, "X_train_tabicl")
print(f"y_train_tabicl memory usage: {y_train_tabicl.memory_usage(deep=True) / (1024 ** 2):.2f} MB")
print_ram_usage("After TabICL context prep")

Shapes:
X_train: (380461, 49)
y_train: (380461,)
X_test: (83356, 49)
y_test: (83356,)

Target distribution:

Train:
target
0   0.978289
1   0.021711
Name: proportion, dtype: float64

Test:
target
0   0.978766
1   0.021234
Name: proportion, dtype: float64

All checks passed.
After loading train/test RAM used: 1.71 GB
X_train memory usage: 71.12 MB
X_test memory usage: 15.58 MB
y_train memory usage: 0.36 MB
y_test memory usage: 0.08 MB


In [ ]:
tabicl_full = TabICLClassifier(
    n_estimators=1,
    batch_size=1,
    kv_cache="repr",
    device="cpu",
    n_jobs=1,
    use_amp=False,
    use_fa3=False,
    offload_mode="disk",
    disk_offload_dir=TABICL_OFFLOAD_DIR,
    random_state=42,
    verbose=True
)

print("Using repr cache + chunked test scoring to avoid kernel crashes.")
print("On CPU/MPS, TabICL does not batch training rows internally, so the training context is capped.")
print(f"Training context rows: {len(X_train_tabicl):,}/{len(X_train):,}")
print(f"Predict chunk size: {TABICL_CHUNK_SIZE:,} rows")
print("TabICL's batch_size only batches ensemble members, not test rows.")
print(f"Disk offload directory: {TABICL_OFFLOAD_DIR}")
print_ram_usage("Before full fit")
tabicl_full.fit(X_train_tabicl, y_train_tabicl)
print_ram_usage("After full fit")
report_cache_sizes(tabicl_full)

In [ ]:
y_test_proba_full = predict_positive_proba_in_chunks(
    tabicl_full,
    X_test,
    chunk_size=TABICL_CHUNK_SIZE
)
y_test_pred_full = (y_test_proba_full >= 0.5).astype(np.int8)

full_results = pd.DataFrame([
    compute_metrics(
        f"TabICLv2_context_{len(X_train_tabicl)}_chunked",
        y_test,
        y_test_pred_full,
        y_test_proba_full
    )
])

full_cm = confusion_matrix(y_test, y_test_pred_full)

print("\n=== FULL RESULTS ===")
display(full_results)

print("\n=== FULL CONFUSION MATRIX ===")
print(full_cm)

print_ram_usage("After full predict")

In [ ]:
for name in ["tabicl_full", "y_test_proba_full", "y_test_pred_full", "full_results", "full_cm"]:
    if name in globals():
        del globals()[name]

gc.collect()

print_ram_usage("After full cleanup")

### **Optional 4-estimator configuration (memory-heavy)**

In [ ]:
RUN_MEMORY_HEAVY_DEFAULT = False

if RUN_MEMORY_HEAVY_DEFAULT:
    tabicl_default = TabICLClassifier(
        n_estimators=4,
        batch_size=1,
        kv_cache="repr",
        device="cpu",
        n_jobs=1,
        use_amp=False,
        use_fa3=False,
        offload_mode="disk",
        disk_offload_dir=TABICL_OFFLOAD_DIR,
        random_state=42,
        verbose=True
    )

    print("Running the 4-estimator configuration. This is much slower and uses much more cache memory.")
    print(f"Training context rows: {len(X_train_tabicl):,}/{len(X_train):,}")
    print_ram_usage("Before default fit")
    tabicl_default.fit(X_train_tabicl, y_train_tabicl)
    print_ram_usage("After default fit")
    report_cache_sizes(tabicl_default)

    y_test_proba_default = predict_positive_proba_in_chunks(
        tabicl_default,
        X_test,
        chunk_size=max(128, TABICL_CHUNK_SIZE // 2)
    )
    y_test_pred_default = (y_test_proba_default >= 0.5).astype(np.int8)

    tabicl_default_results = pd.DataFrame([
        compute_metrics(
            f"TabICLv2_default_context_{len(X_train_tabicl)}_chunked",
            y_test,
            y_test_pred_default,
            y_test_proba_default
        )
    ])

    tabicl_default_cm = confusion_matrix(y_test, y_test_pred_default)

    pd.set_option("display.max_columns", None)
    pd.set_option("display.float_format", "{:.6f}".format)

    print("\n=== TABICLv2 DEFAULT TEST RESULTS ===")
    display(tabicl_default_results)

    print("\n=== CONFUSION MATRIX ===")
    print(tabicl_default_cm)
else:
    print("Skipped the 4-estimator default run.")
    print("Why: on this dataset it commonly exhausts memory even before prediction finishes.")
    print("Set RUN_MEMORY_HEAVY_DEFAULT = True only on a higher-memory machine.")

In [ ]:
import psutil, os

process = psutil.Process(os.getpid())
print(f"RAM used by process: {process.memory_info().rss / 1024**3:.2f} GB")